# Task 1 – Building a Complete ETL Pipeline

ETL stands for:
- Extract – read raw data and validate it
- Transform – clean and enrich the data
- Load – store the cleaned data for analytics

The pipeline will:
1. Generate raw order data (including bad records)
2. Extract and validate records
3. Transform the data into a clean dataset
4. Load the final dataset into a Parquet file

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime

Step 1 — Generate raw data: Create a list of 200 raw order records as dictionaries with these fields: order_id, customer_id, product_name, quantity, unit_price, order_date, shipping_country. Intentionally include at least 15 problematic records across these categories:

In [2]:
products = ["Laptop", "Phone", "Tablet", "Headphones", "Camera"]
countries = ["usa", "canada", "germany", "france", "japan"]

raw_orders = []

for i in range(185):
    order = {
        "order_id": i,
        "customer_id": random.randint(1, 50),
        "product_name": random.choice(products),
        "quantity": random.randint(1, 5),
        "unit_price": round(random.uniform(5, 200), 2),
        "order_date": "2025-01-" + str(random.randint(1,28)),
        "shipping_country": random.choice(countries)
    }
    raw_orders.append(order)




3 records with missing product_name (None or empty string), 
3 records with negative quantity or unit_price, 
3 records with malformed order_date (e.g., "not-a-date", "2025-13-45"), 
3 records with duplicate order_id values, 
3 records where quantity or unit_price is a string instead of a number

In [3]:
for i in range(3):
    raw_orders.append({"order_id": 1000 + i, "customer_id": 1, "product_name": None, "quantity": 2, "unit_price": 20, "order_date": "2025-01-10", "shipping_country": "usa"})
    raw_orders.append({"order_id": 2000 + i, "customer_id": 2, "product_name": "Laptop", "quantity": -1, "unit_price": 50, "order_date": "2025-01-10", "shipping_country": "usa"})
    raw_orders.append({"order_id": 3000 + i, "customer_id": 3, "product_name": "Phone", "quantity": 1, "unit_price": 100, "order_date": "not-a-date", "shipping_country": "usa"})
    raw_orders.append({"order_id": 4000 + i, "customer_id": 4, "product_name": "Tablet", "quantity": "two", "unit_price": "ten", "order_date": "2025-01-15", "shipping_country": "usa"})

raw_orders.append(raw_orders[0])
raw_orders.append(raw_orders[1])
raw_orders.append(raw_orders[2])

print("Total raw records:", len(raw_orders))

Total raw records: 200


Step 2 — Extract with validation: Write an extract(raw_records) function that:

Parses each record and validates required fields exist
Validates that quantity and unit_price are positive numbers
Validates that order_date is a parseable date
Returns two lists: valid_records and rejected_records
Each rejected record should include the reason for rejection
Print a summary: count of valid, count of rejected by reason

In [4]:
def extract(records):
    valid = []
    rejected = []

    for r in records:
        reason = None
        if r["product_name"] is None or r["product_name"] == "":
            reason = "missing name"
        elif not isinstance(r["quantity"], (int, float)) or not isinstance(r["unit_price"], (int, float)):
            reason = "not numeric"
        elif r["quantity"] <= 0 or r["unit_price"] <= 0:
            reason = "negative values"
        elif pd.to_datetime(r["order_date"], errors="coerce") is pd.NaT:
            reason = "invalid date"
            
        if reason is None:
            valid.append(r)
        else:
            rejected.append(r)

    return valid, rejected

valid_records, rejected_records = extract(raw_orders)
print("Valid:", len(valid_records))
print("Rejected:", len(rejected_records))

Valid: 188
Rejected: 12


We separate records into:

valid records → good data and rejected records → bad data

reason = None --> If a problem appears, we store why the record failed.

Reject if: --> product name is None or product name is empty

elif not isinstance(r["quantity"], (int, float)): --> This checks if quantity is a number.

If no problem: --> valid.append(r)
If problem: --> rejected.append(r)

We also store the reason for rejection.

Step 3 — Transform: Write a transform(valid_records) function that:

Converts to a pandas DataFrame

Computes total_amount = quantity * unit_price

Extracts order_month and order_day_of_week from the date

Standardizes shipping_country to title case

Removes duplicate order_id values (keep the first)
|
Adds an amount_category column: "small" (<$25), "medium" ($25-$100), "large" (>$100)

Returns the cleaned DataFrame

In [5]:
def transform(records):
    df = pd.DataFrame(records)
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["total_amount"] = df["quantity"] * df["unit_price"]
    df["order_month"] = df["order_date"].dt.month
    df["shipping_country"] = df["shipping_country"].str.title()
    df = df.drop_duplicates(subset="order_id")

    def get_category(amt):
        if amt < 25: return "small"
        if amt <= 100: return "medium"
        return "large"

    df["amount_category"] = df["total_amount"].apply(get_category)
    return df

clean_df = transform(valid_records)
clean_df.head()

,order_id,customer_id,product_name,quantity,unit_price,order_date,shipping_country,total_amount,order_month,amount_category
0,0,22,Tablet,4,8.38,2025-01-06,France,33.52,1,medium
1,1,5,Laptop,2,82.74,2025-01-02,Usa,165.48,1,large
2,2,9,Tablet,3,154.72,2025-01-07,Canada,464.16,1,large
3,3,37,Phone,3,50.80,2025-01-14,Usa,152.40,1,large
4,4,40,Phone,2,107.69,2025-01-04,Germany,215.38,1,large


Step 4 — Load: Write a load(df, path) function that saves the DataFrame to a Parquet file. Then read it back and verify the row count and dtypes match.

In [6]:
def load(df, path):
    df.to_parquet(path)
    loaded = pd.read_parquet(path)
    return loaded

loaded_df = load(clean_df, "orders_clean.parquet")
print("Rows loaded:", len(loaded_df))

Rows loaded: 185


The load function:
Saves the dataset as Parquet
Reads it back

Verifies:
row count
column types
This ensures data was stored correctly.

Step 5: Run the full pipeline: extract → transform → load. Print a summary showing records at each stage (raw → valid → transformed → loaded).

In [7]:
valid_records, rejected_records = extract(raw_orders)

clean_df = transform(valid_records)

loaded_df = load(clean_df, "orders_clean.parquet")

print("Pipeline Summary")

print("Raw records:", len(raw_orders))
print("Valid records:", len(valid_records))
print("Rejected records:", len(rejected_records))
print("Final dataset rows:", len(loaded_df))

Pipeline Summary
Raw records: 200
Valid records: 188
Rejected records: 12
Final dataset rows: 185


# Task 2 – ETL vs ELT

In ETL (Extract → Transform → Load)  data is cleaned before it is stored.
In ELT ( Extract → Load → Transform) raw data is stored first and cleaned later during analysis. This is common in modern data lakes and cloud warehouses.

Step 1: Implement an ELT variant of the pipeline from Task 1:

Extract: do minimal validation (only reject unparseable JSON/records)
Load: save the raw (mostly unvalidated) data to a Parquet "data lake" file
Transform: read from the data lake file, apply the same validation and transformation rules as Task 1

In [8]:
def elt_extract(raw_orders):
    if isinstance(raw_orders, list):
        return raw_orders
    return []

This extract step is minimal.

It only checks if the record is a dictionary.

We do not validate fields yet

In [9]:
def elt_load(data, path):

    df_raw = pd.DataFrame(data)

    df_raw = df_raw.astype(str)

    df_raw.to_parquet(path, index=False)

    return pd.read_parquet(path)

This function converts raw records into a DataFrame, saves them as a Parquet file

In [10]:
def elt_transform(df_lake):

    df_lake["quantity_numeric"] = pd.to_numeric(df_lake["quantity"], errors="coerce")
    df_lake["unit_price_numeric"] = pd.to_numeric(df_lake["unit_price"], errors="coerce")
    df_lake["order_date_dt"] = pd.to_datetime(df_lake["order_date"], errors="coerce")

    clean_df = df_lake[
        (df_lake["product_name"] != "") &
        (df_lake["product_name"].notna()) &
        (df_lake["quantity_numeric"] > 0) &
        (df_lake["unit_price_numeric"] > 0) &
        (df_lake["order_date_dt"].notna())
    ].copy()

    clean_df = clean_df.drop_duplicates(subset="order_id", keep="first")

    clean_df["total_amount"] = clean_df["quantity_numeric"] * clean_df["unit_price_numeric"]

    clean_df["order_month"] = clean_df["order_date_dt"].dt.month

    clean_df["shipping_country"] = clean_df["shipping_country"].str.title()

    return clean_df

This step cleans and prepares the raw data after it has been loaded into the data lake.
The code converts quantity and unit_price to numeric values and converts order_date to a datetime format. Invalid values are automatically turned into missing values.
Next, the dataset is filtered to keep only valid records with a product name, positive quantity and price, and a valid date.
Duplicate orders are removed using order_id.
Finally, new features are created, including total_amount, order_month, and a standardized shipping_country column.

In [11]:
raw_to_load = elt_extract(raw_orders)

lake_data = elt_load(raw_to_load, "data_lake_raw.parquet")

final_elt_df = elt_transform(lake_data)

print("ELT Final Result:", len(final_elt_df), "records")

ELT Final Result: 188 records


Step 2: Compare ETL and ELT approaches. Write a markdown cell addressing:

How many records made it to the destination in each approach?
At what stage were problems caught in each?
What are the advantages of each approach?
When would you choose one over the other?

In the ETL pipeline, data is validated and transformed before it is loaded into storage.

In the ELT pipeline, raw data is loaded first and the cleaning happens later during the transformation stage.

ETL catches errors earlier and prevents invalid data from entering storage.

ELT keeps raw data available for future analysis and allows transformations to be applied later.

ETL is useful when strict data validation is required before storage.  
ELT is commonly used in modern data lakes and large-scale analytics systems.

## Task 3: Simulate Modes of Dataflow
Model a simplified analytics system with three components: an order ingestion service, a feature computation service, and a prediction service.

Step 1 – Data Passing Through a Database

In this approach, services communicate through a shared database. 
The order service writes orders to the database, the feature service reads the orders and computes features, and the prediction service reads the computed features.

In [12]:
database = {
    "orders": [],
    "features": {}
}

def order_service_write(order):
    database["orders"].append(order)


def feature_service_compute():
    
    df = pd.DataFrame(database["orders"])
    
    if len(df) == 0:
        return
    
    features = {}

    for customer_id, group in df.groupby("customer_id"):
        
        total_orders = len(group)
        avg_amount = group["total_amount"].mean()
        last_order = group["order_date"].max()

        features[customer_id] = {
            "total_orders": total_orders,
            "avg_amount": avg_amount,
            "last_order_date": last_order
        }

    database["features"] = features


def prediction_service_read(customer_id):
    
    return database["features"].get(customer_id, "No features found")

Step 2 – Data Passing Through Services

In this model, services communicate directly with each other. 
The prediction service requests features from the feature service, which in turn requests orders from the order service.

In [13]:
orders_service_data = []

def order_service_add(order):
    orders_service_data.append(order)


def feature_service_get(customer_id):

    df = pd.DataFrame(orders_service_data)

    if len(df) == 0:
        return None

    customer_orders = df[df["customer_id"] == customer_id]

    if len(customer_orders) == 0:
        return None

    return {
        "total_orders": len(customer_orders),
        "avg_amount": customer_orders["total_amount"].mean(),
        "last_order_date": customer_orders["order_date"].max()
    }


def prediction_service_request(customer_id):

    return feature_service_get(customer_id)

Step 3 – Data Passing Through a Message Broker

In this approach, services communicate through a message broker. 
The order service publishes orders, the feature service subscribes and computes features, and the prediction service reads the results.

In [14]:
class Broker:

    def __init__(self):
        self.topics = {}

    def publish(self, topic, message):

        if topic not in self.topics:
            self.topics[topic] = []

        self.topics[topic].append(message)

    def get_latest(self, topic):

        if topic not in self.topics:
            return None

        return self.topics[topic][-1]


broker = Broker()


def order_publish(order):
    broker.publish("orders", order)


def feature_subscribe_and_compute():

    order = broker.get_latest("orders")

    if order is None:
        return

    feature = {
        "customer_id": order["customer_id"],
        "amount": order["total_amount"]
    }

    broker.publish("features", feature)


def prediction_subscribe():

    return broker.get_latest("features")

Step 4: Run 20 new orders through all three dataflow modes. Verify that the prediction service gets the same features in each case. Write a markdown cell comparing the three modes: coupling, latency characteristics, and what happens when one component goes down.

In [15]:
sample_orders = final_elt_df.head(20).to_dict("records")

# DATABASE MODE
for o in sample_orders:
    order_service_write(o)

feature_service_compute()

print("Database mode example:")
print(prediction_service_read(sample_orders[0]["customer_id"]))


# SERVICES MODE
for o in sample_orders:
    order_service_add(o)

print("\nService mode example:")
print(prediction_service_request(sample_orders[0]["customer_id"]))


# BROKER MODE
for o in sample_orders:
    order_publish(o)
    feature_subscribe_and_compute()

print("\nBroker mode example:")
print(prediction_subscribe())

Database mode example:
{'total_orders': 1, 'avg_amount': np.float64(33.52), 'last_order_date': '2025-01-6'}

Service mode example:
{'total_orders': 1, 'avg_amount': np.float64(33.52), 'last_order_date': '2025-01-6'}

Broker mode example:
{'customer_id': '6', 'amount': 430.4}


## TASK 4

Step 1 — Batch processing: Using the cleaned data from Task 1, write a batch processing function that computes daily aggregate features:

Total orders per day
Total revenue per day
Average order size per day
Number of unique customers per day
Top product per day (by revenue)
Execute it and display the results.

In [16]:
df_batch = clean_df.copy()

df_batch["order_date"] = pd.to_datetime(df_batch["order_date"])

df_batch["day"] = df_batch["order_date"].dt.date

In [17]:
batch_summary = df_batch.groupby("day").agg(
    total_orders=("order_id", "count"),
    total_revenue=("total_amount", "sum"),
    avg_order_size=("total_amount", "mean"),
    unique_customers=("customer_id", "nunique")
)

batch_summary

,total_orders,total_revenue,avg_order_size,unique_customers
day,,,,
2025-01-01,3,1015.97,338.656667,3
2025-01-02,11,2024.77,184.070000,10
2025-01-03,5,2953.06,590.612000,5
2025-01-04,7,2350.10,335.728571,6
2025-01-05,5,1605.72,321.144000,5
2025-01-06,2,184.94,92.470000,2
2025-01-07,7,3438.62,491.231429,7
2025-01-08,7,2174.28,310.611429,6
2025-01-09,9,2835.63,315.070000,8


In [18]:
top_products = (
    df_batch
    .groupby(["day", "product_name"])["total_amount"]
    .sum()
    .reset_index()
)

top_products = top_products.sort_values(
    ["day", "total_amount"],
    ascending=[True, False]
)

top_products = top_products.groupby("day").first()

top_products

,product_name,total_amount
day,,
2025-01-01,Headphones,690.17
2025-01-02,Tablet,885.55
2025-01-03,Laptop,2591.38
2025-01-04,Laptop,885.54
2025-01-05,Laptop,709.84
2025-01-06,Phone,151.42
2025-01-07,Headphones,1490.56
2025-01-08,Phone,748.16
2025-01-09,Headphones,1075.96


Step 2 — Stream Processing

Stream processing analyzes data one record at a time.

The system keeps running statistics such as:
- running total of orders
- running revenue
- window average (last 50 orders)
- unique customers
- most popular product in last 50 orders

In [19]:
from collections import deque

In [20]:
class StreamProcessor:

    def __init__(self):
        
        self.total_orders = 0
        self.total_revenue = 0
        
        self.last_50 = deque(maxlen=50)
        
        self.customers = set()
        self.products = []

    
    def process(self, order):
        
        self.total_orders += 1
        self.total_revenue += order["total_amount"]
        
        self.last_50.append(order["total_amount"])
        
        self.customers.add(order["customer_id"])
        
        self.products.append(order["product_name"])
    
    
    def stats(self):
        
        if len(self.last_50) > 0:
            window_avg = sum(self.last_50) / len(self.last_50)
        else:
            window_avg = 0
        
        if len(self.products) > 0:
            popular_product = max(set(self.products), key=self.products.count)
        else:
            popular_product = None
        
        return {
            "total_orders": self.total_orders,
            "total_revenue": self.total_revenue,
            "window_avg": window_avg,
            "unique_customers": len(self.customers),
            "popular_product": popular_product
        }

In [21]:
processor = StreamProcessor()

records = clean_df.to_dict("records")

for i, order in enumerate(records, start=1):
    
    processor.process(order)
    
    if i % 50 == 0:
        print("After", i, "orders")
        print(processor.stats())
        print()

After 50 orders
{'total_orders': 50, 'total_revenue': 15009.890000000005, 'window_avg': 300.1978, 'unique_customers': 35, 'popular_product': 'Camera'}

After 100 orders
{'total_orders': 100, 'total_revenue': 32969.48999999999, 'window_avg': 359.19199999999995, 'unique_customers': 45, 'popular_product': 'Laptop'}

After 150 orders
{'total_orders': 150, 'total_revenue': 54280.230000000025, 'window_avg': 426.2148, 'unique_customers': 48, 'popular_product': 'Laptop'}



Step 3 — Compare Batch and Stream Results

Batch processing analyzes the entire dataset at once and produces accurate historical summaries.

Stream processing works continuously and updates statistics as new data arrives. Windowed metrics (like last 50 orders) may differ from batch results because they only consider recent data.

Batch is useful for historical analysis, while streaming is useful for real-time monitoring and predictions.

## TASK 5

Step 1 — Feature Types

We compute two types of features.

Batch features (from all historical data):
- total_lifetime_orders
- avg_order_amount
- days_since_first_order
- most_purchased_category

Streaming features (last 10 orders):
- recent_order_count
- recent_avg_amount
- seconds_since_last_order
- recent_top_category

In [22]:
customers = clean_df["customer_id"].unique()[:5]

customers

array([22,  5,  9, 37, 40])

In [23]:
batch_features = {}

for customer in customers:
    
    customer_df = clean_df[clean_df["customer_id"] == customer]
    
    total_orders = len(customer_df)
    
    avg_amount = customer_df["total_amount"].mean()
    
    first_order = pd.to_datetime(customer_df["order_date"]).min()
    
    days_since_first = (pd.Timestamp.now() - first_order).days
    
    top_product = customer_df["product_name"].mode()[0]
    
    batch_features[customer] = {
        "total_lifetime_orders": total_orders,
        "avg_order_amount": avg_amount,
        "days_since_first_order": days_since_first,
        "most_purchased_category": top_product
    }

In [24]:
stream_features = {}

for customer in customers:
    
    customer_df = clean_df[clean_df["customer_id"] == customer]
    
    recent = customer_df.tail(10)
    
    recent_count = len(recent)
    
    recent_avg = recent["total_amount"].mean()
    
    last_order = pd.to_datetime(recent["order_date"]).max()
    
    seconds_since = (pd.Timestamp.now() - last_order).total_seconds()
    
    recent_top = recent["product_name"].mode()[0]
    
    stream_features[customer] = {
        "recent_order_count": recent_count,
        "recent_avg_amount": recent_avg,
        "seconds_since_last_order": seconds_since,
        "recent_top_category": recent_top
    }

In [25]:
combined_features = {}

for customer in customers:
    
    combined = {}
    
    combined.update(batch_features[customer])
    combined.update(stream_features[customer])
    
    combined_features[customer] = combined

combined_features

{np.int64(22): {'total_lifetime_orders': 2,
  'avg_order_amount': np.float64(198.45),
  'days_since_first_order': 429,
  'most_purchased_category': 'Laptop',
  'recent_order_count': 2,
  'recent_avg_amount': np.float64(198.45),
  'seconds_since_last_order': 36891079.975612,
  'recent_top_category': 'Laptop'},
 np.int64(5): {'total_lifetime_orders': 4,
  'avg_order_amount': np.float64(226.03),
  'days_since_first_order': 430,
  'most_purchased_category': 'Camera',
  'recent_order_count': 4,
  'recent_avg_amount': np.float64(226.03),
  'seconds_since_last_order': 35422279.981365,
  'recent_top_category': 'Camera'},
 np.int64(9): {'total_lifetime_orders': 5,
  'avg_order_amount': np.float64(327.52799999999996),
  'days_since_first_order': 425,
  'most_purchased_category': 'Tablet',
  'recent_order_count': 5,
  'recent_avg_amount': np.float64(327.52799999999996),
  'seconds_since_last_order': 36113479.987438,
  'recent_top_category': 'Tablet'},
 np.int64(37): {'total_lifetime_orders': 7,
 